# 🛡️ Détection de Spam
**Projet PFA — 3 Modèles ML**
- Modèle 1 : TF-IDF + Régression Logistique (PyTorch)
- Modèle 2 : GRU (PyTorch)
- Modèle 3 : BERT Fine-tuné (PyTorch natif)


## 📦 Installation des dépendances

In [ ]:
!pip install torch transformers scikit-learn nltk pandas numpy matplotlib seaborn wordcloud -q

## 1️⃣ Import & Chargement des données

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device utilisé : {device}")

In [ ]:

path = "./spam_ham_dataset.csv"
df = pd.read_csv(path)
print(df.shape)
df.head()

## 2️⃣ Nettoyage & Prétraitement

In [ ]:
# Suppression colonnes inutiles
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

# Suppression doublons
print(f"Avant déduplication : {df.shape}")
df = df.drop_duplicates()
print(f"Après déduplication : {df.shape}")

# Extraction du sujet
subjects = []
for i in range(len(df)):
    ln = df["text"].iloc[i]
    line = ""
    for ch in ln:
        if ch == '\r':
            break
        line += ch
    line = line.replace("Subject", "")
    subjects.append(line)

df['Subject'] = subjects
df = df.rename(columns={'text': 'Email_text', 'label_num': 'Labels'})
if 'label' in df.columns:
    df = df.drop('label', axis=1)

df['Email_text'] = df['Email_text'].str.lower()
df['Subject'] = df['Subject'].str.lower()

df.head()

In [ ]:
# Visualisation de la distribution
counts = df['Labels'].value_counts()
plt.figure(figsize=(6,4))
sns.barplot(x=['Ham (0)', 'Spam (1)'], y=counts.values, palette=['steelblue','tomato'])
plt.title('Distribution Ham / Spam')
plt.ylabel('Nombre d\'emails')
plt.tight_layout()
plt.show()
print(counts)

In [ ]:
# Prétraitement texte
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

print("Prétraitement en cours...")
df['clean_text'] = df['Email_text'].apply(preprocess_text)
print("✅ Prétraitement terminé")
df[['Email_text', 'clean_text', 'Labels']].head()

In [ ]:
# Split commun pour tous les modèles
X = df['clean_text'].values
y = df['Labels'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

---
# 🔵 MODÈLE 1 — TF-IDF + Régression Logistique (PyTorch)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import issparse

# TF-IDF vectorisation
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

# Conversion en tenseurs denses
X_train_dense = torch.FloatTensor(X_train_tfidf.toarray())
X_test_dense  = torch.FloatTensor(X_test_tfidf.toarray())
y_train_t     = torch.FloatTensor(y_train)
y_test_t      = torch.FloatTensor(y_test)

print(f"Shape TF-IDF train : {X_train_dense.shape}")

In [ ]:
# Dataset PyTorch pour TF-IDF
class TFIDFDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels   = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

train_tfidf_ds = TFIDFDataset(X_train_dense, y_train_t)
test_tfidf_ds  = TFIDFDataset(X_test_dense,  y_test_t)

train_tfidf_loader = DataLoader(train_tfidf_ds, batch_size=64, shuffle=True)
test_tfidf_loader  = DataLoader(test_tfidf_ds,  batch_size=64, shuffle=False)

In [ ]:
# Modèle Régression Logistique en PyTorch
class LogisticRegressionPyTorch(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegressionPyTorch, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x)).squeeze(1)

input_dim = X_train_dense.shape[1]
lr_model  = LogisticRegressionPyTorch(input_dim).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(lr_model.parameters(), lr=0.001, weight_decay=1e-4)

print(lr_model)

In [ ]:
# Boucle d'entraînement
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct = 0, 0
    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = (outputs >= 0.5).float()
        correct += (preds == labels).sum().item()
    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset)
    return avg_loss, accuracy

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = (outputs >= 0.5).float()
            correct += (preds == labels).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset)
    return avg_loss, accuracy, np.array(all_preds), np.array(all_labels)

# Entraînement
EPOCHS_LR = 20
train_losses_lr, val_losses_lr = [], []

for epoch in range(EPOCHS_LR):
    t_loss, t_acc = train_epoch(lr_model, train_tfidf_loader, criterion, optimizer)
    v_loss, v_acc, _, _ = evaluate(lr_model, test_tfidf_loader, criterion)
    train_losses_lr.append(t_loss)
    val_losses_lr.append(v_loss)
    if (epoch+1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS_LR}] | Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f} Acc: {v_acc:.4f}")

print("\n✅ Entraînement Modèle 1 terminé !")

In [ ]:
# Évaluation Modèle 1
_, _, preds_lr, labels_lr = evaluate(lr_model, test_tfidf_loader, criterion)

print("\n📊 RÉSULTATS — Modèle 1 : TF-IDF + Régression Logistique (PyTorch)")
print("-" * 65)
print(f"Accuracy  : {accuracy_score(labels_lr, preds_lr):.4f}")
print(f"Precision : {precision_score(labels_lr, preds_lr):.4f}")
print(f"Recall    : {recall_score(labels_lr, preds_lr):.4f}")
print(f"F1-Score  : {f1_score(labels_lr, preds_lr):.4f}")
print("-" * 65)
print(classification_report(labels_lr, preds_lr, target_names=['Ham', 'Spam']))

# Courbe de loss
plt.figure(figsize=(8,4))
plt.plot(train_losses_lr, label='Train Loss')
plt.plot(val_losses_lr, label='Val Loss')
plt.title('Modèle 1 — Courbe de Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de confusion Modèle 1
cm = confusion_matrix(labels_lr, preds_lr)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham','Spam'], yticklabels=['Ham','Spam'])
plt.title('Matrice de confusion — Modèle 1')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.show()

---
# 🟡 MODÈLE 2 — GRU (PyTorch)

In [ ]:
from collections import Counter

# Construction du vocabulaire
VOCAB_SIZE = 10000
MAX_LEN    = 100

# Tokenisation simple
def tokenize(text):
    return text.split()

# Construction du vocabulaire depuis train
all_words = []
for text in X_train:
    all_words.extend(tokenize(text))

word_counts = Counter(all_words)
vocab = ['<PAD>', '<UNK>'] + [w for w, _ in word_counts.most_common(VOCAB_SIZE - 2)]
word2idx = {w: i for i, w in enumerate(vocab)}

print(f"Taille du vocabulaire : {len(vocab)}")

def encode(text, max_len):
    tokens = tokenize(text)[:max_len]
    ids = [word2idx.get(w, 1) for w in tokens]  # 1 = <UNK>
    # Padding
    ids += [0] * (max_len - len(ids))
    return ids

# Encodage
X_train_seq = [encode(t, MAX_LEN) for t in X_train]
X_test_seq  = [encode(t, MAX_LEN) for t in X_test]

print(f"Exemple encodage : {X_train_seq[0][:10]}")

In [ ]:
# Dataset PyTorch pour GRU
class SequenceDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.LongTensor(sequences)
        self.labels    = torch.FloatTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

train_seq_ds = SequenceDataset(X_train_seq, y_train)
test_seq_ds  = SequenceDataset(X_test_seq,  y_test)

train_seq_loader = DataLoader(train_seq_ds, batch_size=64, shuffle=True)
test_seq_loader  = DataLoader(test_seq_ds,  batch_size=64, shuffle=False)

In [ ]:
# Architecture GRU en PyTorch
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.3):
        super(GRUClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x : (batch, seq_len)
        embedded = self.embedding(x)           # (batch, seq_len, embed_dim)
        _, hidden = self.gru(embedded)         # hidden : (num_layers, batch, hidden_dim)
        hidden = self.dropout(hidden[-1])      # Dernier layer : (batch, hidden_dim)
        out = torch.sigmoid(self.fc(hidden))   # (batch, 1)
        return out.squeeze(1)

# Hyperparamètres
EMBED_DIM  = 128
HIDDEN_DIM = 64
NUM_LAYERS = 2
DROPOUT    = 0.3

gru_model = GRUClassifier(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

print(gru_model)
total_params = sum(p.numel() for p in gru_model.parameters())
print(f"\nNombre total de paramètres : {total_params:,}")

In [ ]:
# Entraînement GRU
gru_criterion = nn.BCELoss()
gru_optimizer = torch.optim.Adam(gru_model.parameters(), lr=0.001)
gru_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(gru_optimizer, patience=2, factor=0.5)

EPOCHS_GRU = 10
train_losses_gru, val_losses_gru = [], []
best_val_loss = float('inf')
best_gru_state = None

for epoch in range(EPOCHS_GRU):
    t_loss, t_acc = train_epoch(gru_model, train_seq_loader, gru_criterion, gru_optimizer)
    v_loss, v_acc, _, _ = evaluate(gru_model, test_seq_loader, gru_criterion)
    gru_scheduler.step(v_loss)
    train_losses_gru.append(t_loss)
    val_losses_gru.append(v_loss)

    # Sauvegarde du meilleur modèle
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_gru_state = gru_model.state_dict().copy()

    print(f"Epoch [{epoch+1}/{EPOCHS_GRU}] | Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f} Acc: {v_acc:.4f}")

# Charger le meilleur état
gru_model.load_state_dict(best_gru_state)
print("\n✅ Entraînement Modèle 2 terminé !")

In [ ]:
# Évaluation Modèle 2
_, _, preds_gru, labels_gru = evaluate(gru_model, test_seq_loader, gru_criterion)

print("\n📊 RÉSULTATS — Modèle 2 : GRU (PyTorch)")
print("-" * 65)
print(f"Accuracy  : {accuracy_score(labels_gru, preds_gru):.4f}")
print(f"Precision : {precision_score(labels_gru, preds_gru):.4f}")
print(f"Recall    : {recall_score(labels_gru, preds_gru):.4f}")
print(f"F1-Score  : {f1_score(labels_gru, preds_gru):.4f}")
print("-" * 65)
print(classification_report(labels_gru, preds_gru, target_names=['Ham', 'Spam']))

# Courbe de loss
plt.figure(figsize=(8,4))
plt.plot(train_losses_gru, label='Train Loss')
plt.plot(val_losses_gru, label='Val Loss')
plt.title('Modèle 2 — Courbe de Loss (GRU)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

# Matrice de confusion
cm = confusion_matrix(labels_gru, preds_gru)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Ham','Spam'], yticklabels=['Ham','Spam'])
plt.title('Matrice de confusion — Modèle 2 (GRU)')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.show()

---
# 🔴 MODÈLE 3 — BERT Fine-tuné (PyTorch natif, sans Trainer)

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Re-split sur textes bruts (non prétraités) — meilleur pour BERT
raw_texts = df['Email_text'].astype(str).values
raw_labels = df['Labels'].values

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    raw_texts, raw_labels, test_size=0.2, random_state=42, stratify=raw_labels
)

print(f"Train BERT: {len(X_train_raw)} | Test BERT: {len(X_test_raw)}")

In [ ]:
# Tokenizer BERT
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

MAX_LEN_BERT = 128

train_enc = bert_tokenizer(
    list(X_train_raw),
    truncation=True,
    padding='max_length',
    max_length=MAX_LEN_BERT,
    return_tensors='pt'
)

test_enc = bert_tokenizer(
    list(X_test_raw),
    truncation=True,
    padding='max_length',
    max_length=MAX_LEN_BERT,
    return_tensors='pt'
)

print("✅ Tokenisation BERT terminée")
print(f"Clés : {list(train_enc.keys())}")

In [ ]:
# Dataset PyTorch pour BERT
class BERTDataset(Dataset):
    def __init__(self, encodings, labels):
        self.input_ids      = encodings['input_ids']
        self.attention_mask = encodings['attention_mask']
        self.token_type_ids = encodings.get('token_type_ids', None)
        self.labels         = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels':         self.labels[idx]
        }
        if self.token_type_ids is not None:
            item['token_type_ids'] = self.token_type_ids[idx]
        return item

train_bert_ds = BERTDataset(train_enc, y_train_raw)
test_bert_ds  = BERTDataset(test_enc,  y_test_raw)

train_bert_loader = DataLoader(train_bert_ds, batch_size=16, shuffle=True)
test_bert_loader  = DataLoader(test_bert_ds,  batch_size=16, shuffle=False)

print(f"Batches train : {len(train_bert_loader)} | Batches test : {len(test_bert_loader)}")

In [ ]:
# Chargement de BERT pour classification binaire
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
).to(device)

# Optimiseur AdamW avec weight decay
EPOCHS_BERT = 3
LR_BERT     = 2e-5

bert_optimizer = AdamW(bert_model.parameters(), lr=LR_BERT, weight_decay=0.01)

total_steps = len(train_bert_loader) * EPOCHS_BERT
scheduler = get_linear_schedule_with_warmup(
    bert_optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(f"Total steps : {total_steps} | Warmup steps : {int(0.1 * total_steps)}")

In [ ]:
# Boucles d'entraînement et d'évaluation BERT
def train_bert_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, correct = 0, 0
    for batch in loader:
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        logits  = outputs.logits

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)


def eval_bert(model, loader):
    model.eval()
    total_loss, correct = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss    = outputs.loss
            logits  = outputs.logits

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), correct / len(loader.dataset), np.array(all_preds), np.array(all_labels)

print("✅ Fonctions d'entraînement BERT définies")

In [ ]:
# Entraînement BERT
train_losses_bert, val_losses_bert = [], []
best_val_loss_bert = float('inf')
best_bert_state = None

for epoch in range(EPOCHS_BERT):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS_BERT} ---")
    t_loss, t_acc = train_bert_epoch(bert_model, train_bert_loader, bert_optimizer, scheduler)
    v_loss, v_acc, _, _ = eval_bert(bert_model, test_bert_loader)

    train_losses_bert.append(t_loss)
    val_losses_bert.append(v_loss)

    if v_loss < best_val_loss_bert:
        best_val_loss_bert = v_loss
        best_bert_state = {k: v.clone() for k, v in bert_model.state_dict().items()}

    print(f"Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f} Acc: {v_acc:.4f}")

bert_model.load_state_dict(best_bert_state)
print("\n✅ Fine-tuning BERT terminé !")

In [ ]:
# Évaluation Modèle 3
_, _, preds_bert, labels_bert = eval_bert(bert_model, test_bert_loader)

print("\n📊 RÉSULTATS — Modèle 3 : BERT Fine-tuné (PyTorch natif)")
print("-" * 65)
print(f"Accuracy  : {accuracy_score(labels_bert, preds_bert):.4f}")
print(f"Precision : {precision_score(labels_bert, preds_bert):.4f}")
print(f"Recall    : {recall_score(labels_bert, preds_bert):.4f}")
print(f"F1-Score  : {f1_score(labels_bert, preds_bert):.4f}")
print("-" * 65)
print(classification_report(labels_bert, preds_bert, target_names=['Ham', 'Spam']))

# Courbe de loss
plt.figure(figsize=(8,4))
plt.plot(train_losses_bert, label='Train Loss')
plt.plot(val_losses_bert, label='Val Loss')
plt.title('Modèle 3 — Courbe de Loss (BERT)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

# Matrice de confusion
cm = confusion_matrix(labels_bert, preds_bert)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Ham','Spam'], yticklabels=['Ham','Spam'])
plt.title('Matrice de confusion — Modèle 3 (BERT)')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.show()

---
# 📊 Comparaison des 3 Modèles

In [ ]:
results = {
    'Modèle': ['TF-IDF + LR (PyTorch)', 'GRU (PyTorch)', 'BERT Fine-tuné'],
    'Accuracy':  [
        accuracy_score(labels_lr, preds_lr),
        accuracy_score(labels_gru, preds_gru),
        accuracy_score(labels_bert, preds_bert)
    ],
    'Precision': [
        precision_score(labels_lr, preds_lr),
        precision_score(labels_gru, preds_gru),
        precision_score(labels_bert, preds_bert)
    ],
    'Recall': [
        recall_score(labels_lr, preds_lr),
        recall_score(labels_gru, preds_gru),
        recall_score(labels_bert, preds_bert)
    ],
    'F1-Score': [
        f1_score(labels_lr, preds_lr),
        f1_score(labels_gru, preds_gru),
        f1_score(labels_bert, preds_bert)
    ]
}

results_df = pd.DataFrame(results)
results_df = results_df.set_index('Modèle')
print(results_df.round(4))

In [ ]:
# Graphique de comparaison
metrics   = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
modeles   = results_df.index.tolist()
x         = np.arange(len(metrics))
width     = 0.25
colors    = ['steelblue', 'orange', 'tomato']

fig, ax = plt.subplots(figsize=(12, 5))
for i, (mod, color) in enumerate(zip(modeles, colors)):
    vals = results_df.loc[mod, metrics].values
    bars = ax.bar(x + i*width, vals, width, label=mod, color=color, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Métriques')
ax.set_ylabel('Score')
ax.set_title('Comparaison des 3 Modèles de Détection de Spam')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0.8, 1.02)
ax.legend()
plt.tight_layout()
plt.show()

---
# 💾 Sauvegarde des Modèles (PyTorch)

In [ ]:
import pickle, os

os.makedirs('saved_models', exist_ok=True)

# Modèle 1 — poids PyTorch + vectoriseur TF-IDF
torch.save(lr_model.state_dict(), 'saved_models/lr_model.pt')
with open('saved_models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Modèle 2 — poids PyTorch + vocabulaire
torch.save(gru_model.state_dict(), 'saved_models/gru_model.pt')
with open('saved_models/vocab.pkl', 'wb') as f:
    pickle.dump(word2idx, f)

# Modèle 3 — BERT complet
bert_model.save_pretrained('saved_models/bert_model')
bert_tokenizer.save_pretrained('saved_models/bert_tokenizer')

print("✅ Tous les modèles sauvegardés dans saved_models/")

---
# 🔮 Inférence — Prédire un nouvel email

In [ ]:
def predict_email(text, model_choice='bert'):
    """
    Prédit si un email est spam ou ham.
    model_choice : 'lr', 'gru', ou 'bert'
    Retourne : {'label': 'Spam'/'Ham', 'probability': float}
    """
    if model_choice == 'lr':
        clean = preprocess_text(text)
        vec   = tfidf.transform([clean]).toarray()
        t     = torch.FloatTensor(vec).to(device)
        lr_model.eval()
        with torch.no_grad():
            prob = lr_model(t).item()

    elif model_choice == 'gru':
        clean = preprocess_text(text)
        seq   = encode(clean, MAX_LEN)
        t     = torch.LongTensor([seq]).to(device)
        gru_model.eval()
        with torch.no_grad():
            prob = gru_model(t).item()

    elif model_choice == 'bert':
        enc = bert_tokenizer(
            text, truncation=True, padding='max_length',
            max_length=MAX_LEN_BERT, return_tensors='pt'
        )
        bert_model.eval()
        with torch.no_grad():
            logits = bert_model(
                input_ids=enc['input_ids'].to(device),
                attention_mask=enc['attention_mask'].to(device)
            ).logits
        probs = torch.softmax(logits, dim=1)
        prob  = probs[0][1].item()  # Probabilité classe Spam

    label = 'Spam 🚫' if prob >= 0.5 else 'Ham ✅'
    return {'label': label, 'probability': round(prob, 4)}


# Test
test_emails = [
    "Congratulations! You've won a free iPhone. Click here to claim your prize now!",
    "Hi, can we schedule a meeting tomorrow at 3pm to discuss the project?",
    "URGENT: Your bank account has been suspended. Verify immediately to avoid charges."
]

print("🔍 Test de prédiction avec les 3 modèles\n")
for email in test_emails:
    print(f"Email : {email[:60]}...")
    for model in ['lr', 'gru', 'bert']:
        result = predict_email(email, model_choice=model)
        print(f"  [{model.upper():4}] {result['label']} | Probabilité spam : {result['probability']:.4f}")
    print()

In [ ]:
import shutil
from google.colab import files

# 1. Compresser le dossier en un fichier .zip
shutil.make_archive('saved_models_archive', 'zip', 'saved_models')

# 2. Télécharger le fichier sur ton PC
files.download('saved_models_archive.zip')